# BankNifty Bot — Colab Sweep

Runs the full **import → resample → sweep → ensemble** pipeline on Colab's free CPU/T4.
Results are saved back to Google Drive so you can load them into the local dashboard.

## Before you start

Upload **one file** to a folder in your Google Drive (e.g. `My Drive/banknifty_bot/`):

| File | What it is |
|---|---|
| `bank-nifty-1m-data.csv` | The 11-year raw 1-minute CSV |

That's it — the repo is cloned automatically from GitHub.

Then **run cells top-to-bottom**.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

## Step 2 — Configuration

Edit the variables below to match your Drive folder. Copy the config block from the
**🧪 Experiment Builder** page in the dashboard to override the sweep settings.

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
DRIVE_FOLDER   = '/content/drive/MyDrive/banknifty_bot'   # Your Drive folder
RAW_CSV        = f'{DRIVE_FOLDER}/bank-nifty-1m-data.csv' # Raw 1m data CSV on Drive

REPO_URL       = 'https://github.com/aditya33agrawal/banknifty-trading-bot'
REPO_BRANCH    = 'main'

# ── Sweep settings — paste from Experiment Builder to override ────────────────
INTERVALS      = '5m,15m,30m,60m,1d'
STRATEGIES     = 'ema_trend,supertrend,rsi_reversion,donchian,orb,vwap'
FOLDS          = 5
MAX_COMBOS     = 60
OBJECTIVE      = 'calmar'    # calmar | sortino | sharpe | profit_factor
SLIPPAGE       = 1.0         # ticks
RISK_PCT       = 1.0         # % of equity per trade
ENSEMBLE_TOP_N = 4           # blend top-N cells after sweep
# ── END CONFIG ───────────────────────────────────────────────────────────────

REPO_DIR  = '/content/banknifty_bot'
OUT_DIR   = f'{REPO_DIR}/outputs'
SWEEP_OUT = f'{OUT_DIR}/sweep_results.json'
ENS_OUT   = f'{OUT_DIR}/ensemble_result.json'
print('Config OK')

## Step 3 — Clone repo and install

In [ ]:
import os, subprocess, shutil

# Remove stale clone if present
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f'Cloning {REPO_URL} (branch: {REPO_BRANCH})…')
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('CLONE ERROR:', result.stderr)
else:
    print(f'Cloned to {REPO_DIR}')
    # List top-level to confirm
    print(os.listdir(REPO_DIR))

In [ ]:
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

result = subprocess.run(
    ['pip', 'install', '-q', '-e', '.[dev]'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('INSTALL ERROR:', result.stderr[-2000:])
else:
    print('Package installed')
    import importlib
    importlib.import_module('banknifty_bot')
    print('banknifty_bot importable')

## Step 4 — Import 1-minute data

*Skip this cell if you already have the processed parquet store on Drive and copied it here.*

In [ ]:
from pathlib import Path

processed_1m = Path('data/processed/symbol=NSEBANK/interval=1m')
if processed_1m.exists() and any(processed_1m.rglob('*.parquet')):
    print('1m parquet partition already exists — skipping import')
else:
    print(f'Importing {RAW_CSV} (takes 1-3 min)…')
    result = subprocess.run(
        ['python', 'scripts/import_bulk_csv.py',
         '--file', RAW_CSV,
         '--interval', '1m',
         '--dayfirst'],
        capture_output=True, text=True
    )
    print(result.stdout[-3000:] if result.stdout else '')
    if result.returncode != 0:
        print('ERROR:', result.stderr[-2000:])
    else:
        print('Import complete')

## Step 5 — Validate data

In [ ]:
result = subprocess.run(['python', 'scripts/check_data.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('WARNING:', result.stderr[-1000:])
    print('\nData issues detected — review above before continuing')
else:
    print('Data validation passed')

## Step 6 — Resample to higher intervals

In [ ]:
target_intervals = [i for i in INTERVALS.split(',') if i != '1d']
target_str = ','.join(target_intervals) if target_intervals else '5m,15m,30m,60m'

print(f'Resampling to: {target_str}')
result = subprocess.run(
    ['python', 'scripts/resample_intervals.py', '--intervals', target_str],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if result.stdout else '')
if result.returncode != 0:
    print('ERROR:', result.stderr[-2000:])
else:
    print('Resample complete')

## Step 7 — Run the sweep

This is the heavy cell. A 5×6 matrix with 5 folds / 60 combos takes **2–4 hours** on Colab CPU.

The sweep is **resumable** — if the Colab session disconnects, re-run this cell
and it will skip already-completed cells (results stream to disk after each cell).

In [ ]:
import sys, json
from datetime import datetime

os.makedirs(OUT_DIR, exist_ok=True)

sweep_cmd = [
    sys.executable, 'scripts/run_sweep.py',
    '--intervals',   INTERVALS,
    '--strategies',  STRATEGIES,
    '--folds',       str(FOLDS),
    '--max-combos',  str(MAX_COMBOS),
    '--objective',   OBJECTIVE,
    '--slippage',    str(SLIPPAGE),
    '--risk-pct',    str(RISK_PCT),
    '--out',         SWEEP_OUT,
]

print('Starting sweep at', datetime.now().strftime('%H:%M:%S'))
print('Command:', ' '.join(sweep_cmd))
print('─' * 60)

proc = subprocess.Popen(sweep_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

print('─' * 60)
if proc.returncode != 0:
    print(f'Sweep exited with code {proc.returncode}')
else:
    n = len(json.loads(Path(SWEEP_OUT).read_text())) if Path(SWEEP_OUT).exists() else 0
    print(f'Sweep complete — {n} results in {SWEEP_OUT}')
    print('Finished at', datetime.now().strftime('%H:%M:%S'))

## Step 8 — Build ensemble from top-N winners

In [ ]:
result = subprocess.run(
    [sys.executable, 'scripts/build_ensemble.py',
     '--top',       str(ENSEMBLE_TOP_N),
     '--objective', OBJECTIVE,
     '--sweep',     SWEEP_OUT,
     '--out',       ENS_OUT],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr[-2000:])
else:
    print('Ensemble built')

## Step 9 — Save results to Drive and download

Results are copied to your Drive folder **and** offered as direct downloads to your browser.
Drop the downloaded JSONs into `outputs/` in the local repo, or upload via the dashboard.

In [ ]:
import shutil
from google.colab import files as colab_files

drive_out = Path(DRIVE_FOLDER) / 'outputs'
drive_out.mkdir(parents=True, exist_ok=True)

to_copy = [SWEEP_OUT, ENS_OUT]
downloaded = []
for src in to_copy:
    p = Path(src)
    if p.exists():
        dst = drive_out / p.name
        shutil.copy2(src, dst)
        print(f'Copied {p.name} → Drive:{dst}')
        downloaded.append(src)
    else:
        print(f'{p.name} not found — did the sweep/ensemble step succeed?')

print('\nDownloading files to your browser…')
for path in downloaded:
    colab_files.download(path)

print('\nNext steps:')
print('  1. Files saved to Drive:', str(drive_out))
print('  2. Files downloaded to your browser.')
print('  3. Open the local dashboard → Results Leaderboard → Upload from Colab / Drive.')

## Quick results preview (in Colab)

In [ ]:
import json, pandas as pd

sweep_path = Path(SWEEP_OUT)
if not sweep_path.exists():
    print('No results yet — run the sweep step first.')
else:
    df = pd.DataFrame(json.loads(sweep_path.read_text()))
    cols = ['interval','strategy','oos_cagr_pct','oos_sharpe','oos_calmar',
            'oos_max_dd_pct','oos_profit_factor','oos_n_trades','gates_passed','mc_prob_profit']
    cols = [c for c in cols if c in df.columns]
    print(f'Total cells: {len(df)}')
    display(df[cols].sort_values('oos_calmar', ascending=False).head(20))